# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [1]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

## Parameters

In [2]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2026, 1, 1)
END_DATE   = datetime.date(2026, 1, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 1, 3)

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [3]:
# Load only the market files whose month falls within [START_DATE, END_DATE].
import datetime as _dt
def _file_in_range(p, start, end):
    """Return True if YYYY-MM.parquet filename falls within the date range."""
    try:
        y, m = (int(x) for x in p.stem.split("-"))
        file_date = _dt.date(y, m, 1)
        return start.replace(day=1) <= file_date <= end.replace(day=1)
    except Exception:
        return False

market_files = [
    p for p in sorted(MARKETS_DIR.glob("*.parquet"))
    if _file_in_range(p, START_DATE, END_DATE)
]
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 1 market file(s)
Unique markets (all):  89,622
Unique markets (filtered 2026-01-01 → 2026-01-05): 12,582


,end_date_iso,condition_id,market_json
0,2026-01-01T00:00:00Z,0x006ce0742c3891c396aead079e563c5d58c4eae2f9b0...,"{""enable_order_book"":false,""active"":true,""clos..."
1,2026-01-01T00:00:00Z,0x008add40355561724afbce56f68f1aa4ca4d83d8bdba...,"{""enable_order_book"":false,""active"":true,""clos..."


In [4]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Token lookup entries: 25,139
Market meta rows:     12,582


## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel.  Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal.  The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

In [5]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.shard_processor import (
    select_top_wallets_shard,
    enrich_and_group_shard,
)

trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade shard(s)")

sample_raw = pd.read_parquet(trade_files[0])
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)

Found 16 trade shard(s)
Sample shard rows (0.parquet): 5,498,880
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']
Date range: 2025-01-01 → 2026-03-11


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,wallet,side,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,7875320516565813053445635107732149656386289192...,No,0x126f15c7bd21bf5bf136f3629e10990dc0677137,BUY,...,1,10.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 20...",BUY,No,7875320516565813053445635107732149656386289192...,0.62,6.2
1,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,5866505527798653441671993940591462145801013733...,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,BUY,...,1,10.0,True,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 20...",SELL,No,7875320516565813053445635107732149656386289192...,0.62,6.2
2,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38...,1531,66159089,1735689997,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,5866505527798653441671993940591462145801013733...,Yes,0x260d1ae03c985f7c78ad286b5da14940b4739679,BUY,...,1,30.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 20...",BUY,Yes,5866505527798653441671993940591462145801013733...,0.37,11.1


In [6]:
# ── build token resolution DataFrame ────────────────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard...")
shard_wallet_pnl: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(select_top_wallets_shard, f, token_df, END_TRAIN_TS): f
        for f in trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnl or pnl > shard_wallet_pnl[w]:
                shard_wallet_pnl[w] = pnl
        if i % 4 == 0 or i == len(trade_files):
            print(
                f"  {i}/{len(trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnl):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnl.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")

Phase 1 — selecting top-4% wallets per shard...
  4/16 shards | raw: 22,850,204 | in-range: 477,293 | wallet union so far: 904
  8/16 shards | raw: 46,359,306 | in-range: 932,724 | wallet union so far: 1,504
  12/16 shards | raw: 69,523,558 | in-range: 1,418,992 | wallet union so far: 1,939
  16/16 shards | raw: 93,357,019 | in-range: 1,995,789 | wallet union so far: 2,371

Phase 1 done. Candidate wallets (union of top-4% per shard): 2,371


## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [7]:
# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training P&L per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(enrich_and_group_shard, f, token_df, END_TRAIN_TS, top_wallets): f
        for f in trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(trade_files):
            print(f"  {i}/{len(trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)

Phase 2 — grouping and filtering shards...
  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 746,361
  Unique wallets in grouped:                   2,371


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,avg_price
0,0xa9e3d5d200b573ea88adabfe2f5825b219873201e75c...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,BUY,2025-12-24 12:49:25+00:00,0x51ce2667ff49e6ccbba8a60880073fbfcbef8596995a...,Bills,True,1.0,111.070988,111.070988,99.999999,111.070988,4,11.070989,0.900325
1,0xc663dbeddb14a51eecc30e98444aef75b28e2d14ec94...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,BUY,2025-12-24 12:49:33+00:00,0xdaa4c05704bd2a091d3f68da5de24f4525e766a86208...,Rams,True,1.0,114.707500,114.707500,100.000000,114.707500,4,14.707500,0.871783
2,0x381fdf761a6f0d70094402ebf1c074816fce5dfc2b33...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,SELL,2025-12-25 20:06:33+00:00,0xdaa4c05704bd2a091d3f68da5de24f4525e766a86208...,Rams,True,1.0,96.707500,114.700000,94.234000,114.700000,3,-20.466000,0.821569
3,0xae937f7726e05b8cce07c8ee62d80bd6e7372b83a60d...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,SELL,2025-12-25 20:06:57+00:00,0x51ce2667ff49e6ccbba8a60880073fbfcbef8596995a...,Bills,True,1.0,87.070988,111.070000,90.206700,111.070000,6,-20.863300,0.812161
4,0x7b5b1a5beedfbf9482028e8284fbee9f98622bd9300d...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,BUY,2025-12-31 13:08:55+00:00,0xe1c1cfe25fa7d982a943264579081767708594952ea5...,Bulls,True,1.0,192.307691,192.307691,99.999999,192.307691,2,92.307692,0.520000


## 4 · Phase 3: apply final PnL cut-off and export

Compute the true cross-shard training P&L per wallet.  Use the **median** of the
per-shard P&L values accumulated in Phase 1 as the export cut-off: wallets whose
total training P&L falls below that median are dropped before writing.

In [8]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",        "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        pnl_usdc        = ("trade_pnl",        "sum"),
    )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard pnl values collected in Phase 1 ─────
import statistics
shard_pnl_values = list(shard_wallet_pnl.values())
pnl_cutoff = statistics.median(shard_pnl_values)

final_wallets = set(
    wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_cutoff, "wallet"]
)

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard PnL median (cut-off):        {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing PnL cut-off:           {len(final_wallets):,}")
wallet_summary.head(5)

Unique wallets after Phase 2 union:    2,371
Per-shard PnL median (cut-off):        1,009.48 USDC
Wallets passing PnL cut-off:           925


,wallet,num_markets,num_trades,total_cost_usdc,pnl_usdc
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,36,2208,2.482478e+06,1.306898e+06
1,0x0d3b10b8eac8b089c6e4a695e65d8e044167c46b,30,8912,6.604998e+06,8.269899e+05
2,0x492442eab586f242b53bda933fd5de859c8a3782,34,1556,3.065441e+06,5.452317e+05
3,0xe74a4446efd66a4de690962938f550d8921a40ee,8,403,6.914630e+05,3.993424e+05
4,0x37e4728b3c4607fb2b3b205386bb1d1fb1a8c991,41,668,1.452030e+06,3.985996e+05


## 5 · Market-level volume summary

In [9]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 3,617


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x386c70eb5b79e8d6b05bc371033387f0b9a5d9318042...,Seahawks vs. 49ers,2026-01-04 00:00:00+00:00,302,14591,6.494799e+06,7.047767e+06
1,0x68730687766f5294a9840b6d3b7b57b8040d082dd69c...,Oregon vs. Texas Tech Red Raiders,2026-01-01 00:00:00+00:00,261,7473,4.673537e+06,4.605661e+06
2,0x39310dbdbc542eda4650a85fb8121fad8235485a4c46...,Thunder vs. Warriors,2026-01-03 00:00:00+00:00,266,8939,4.094173e+06,4.327399e+06
3,0x6fe48420bd2c25a2594d158487b08fbc12edd080ff8c...,Ravens vs. Steelers,2026-01-04 00:00:00+00:00,294,13229,4.066009e+06,3.322754e+06
4,0xa6535047a0fef371d1d1b44693508960eef82c53db3e...,Panthers vs. Buccaneers,2026-01-04 00:00:00+00:00,274,6603,3.754912e+06,3.856833e+06
5,0xc48fcf49b52f5687875d5636f718ff726a18a108c086...,Knicks vs. Spurs,2026-01-01 00:00:00+00:00,331,11216,3.687871e+06,3.724589e+06
6,0x269b70d13789a587fde5b6482688ae5bb18c8de7e357...,Ole Miss vs. Georgia,2026-01-02 00:00:00+00:00,256,11825,3.459044e+06,3.520644e+06
7,0x88a945f5acfae0e9f2ad6af05819554212da4f9e36c1...,Timberwolves vs. Heat,2026-01-03 00:00:00+00:00,712,8457,3.397271e+06,3.569592e+06
8,0xbd3a3e46de3aab7d3180e1981269041dac35dc338029...,Miami vs. Ohio State,2026-01-01 00:00:00+00:00,253,8212,3.157834e+06,3.419066e+06
9,0xf70c920368b689bdd16772f9eda0959708f8cbf18742...,Spread: Indiana (-7.5),2026-01-01 00:00:00+00:00,118,4797,3.066604e+06,3.173286e+06


## 6 · Wallet statistics (quantiles)

In [10]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]    = wallet_stats["pnl_usdc"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc,wallet_count_complement
quantile,,,,,
0.001,2,1.0,1.0,-391264.77,2369
0.01,24,1.0,1.0,-63519.17,2347
0.05,119,2.0,1.0,-8020.97,2252
0.1,237,4.0,1.0,-3175.15,2134
0.25,593,17.0,4.0,-493.52,1778
0.5,1186,29.0,9.0,684.61,1185
0.75,1778,106.0,14.0,1777.44,593
0.9,2134,352.0,31.0,5209.79,237
0.95,2252,774.5,63.0,10733.03,119


## 7 · Full enriched trade table

In [11]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,num_fills
0,0xa9e3d5d200b573ea88adabfe2f5825b219873201e75c...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,BUY,2025-12-24 12:49:25+00:00,0x51ce2667ff49e6ccbba8a60880073fbfcbef8596995a...,Bills,111.070988,111.070988,0.900325,True,1.0,99.999999,111.070988,11.070989,4
1,0xc663dbeddb14a51eecc30e98444aef75b28e2d14ec94...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,BUY,2025-12-24 12:49:33+00:00,0xdaa4c05704bd2a091d3f68da5de24f4525e766a86208...,Rams,114.707500,114.707500,0.871783,True,1.0,100.000000,114.707500,14.707500,4
2,0x381fdf761a6f0d70094402ebf1c074816fce5dfc2b33...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,SELL,2025-12-25 20:06:33+00:00,0xdaa4c05704bd2a091d3f68da5de24f4525e766a86208...,Rams,96.707500,114.700000,0.821569,True,1.0,94.234000,114.700000,-20.466000,3
3,0xae937f7726e05b8cce07c8ee62d80bd6e7372b83a60d...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,SELL,2025-12-25 20:06:57+00:00,0x51ce2667ff49e6ccbba8a60880073fbfcbef8596995a...,Bills,87.070988,111.070000,0.812161,True,1.0,90.206700,111.070000,-20.863300,6
4,0x7b5b1a5beedfbf9482028e8284fbee9f98622bd9300d...,0x0026d53f4c96de6b2f3d77aab49b296333281ed5,BUY,2025-12-31 13:08:55+00:00,0xe1c1cfe25fa7d982a943264579081767708594952ea5...,Bulls,192.307691,192.307691,0.520000,True,1.0,99.999999,192.307691,92.307692,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746356,0x431ead16b09b9f0db0f38637425669b8f2bf2f27d558...,0xff9ca5d2aa3b3e8e4e47d5fa8fd4c30f5f372c98,SELL,2026-01-03 23:42:17+00:00,0x88a945f5acfae0e9f2ad6af05819554212da4f9e36c1...,Heat,0.000000,168.619049,0.420000,False,0.0,70.820001,0.000000,70.820001,1
746357,0x431ead16b09b9f0db0f38637425669b8f2bf2f27d558...,0xff9ca5d2aa3b3e8e4e47d5fa8fd4c30f5f372c98,BUY,2026-01-03 23:42:17+00:00,0x88a945f5acfae0e9f2ad6af05819554212da4f9e36c1...,Timberwolves,1187.990951,1187.990951,0.580000,True,1.0,689.034752,1187.990951,498.956199,1
746358,0x847d2457cb6795e7ad71c1da971b1030078e382a7e3f...,0xff9ca5d2aa3b3e8e4e47d5fa8fd4c30f5f372c98,BUY,2026-01-04 00:17:33+00:00,0x2f3d58528326e679b701627fc852a244e18b5c7ea521...,Knicks,623.980472,274.000000,0.610000,False,0.0,167.140000,0.000000,-167.140000,1
746359,0xbf7be97052e226ed0a515af7cfdd2d2d5033c9755904...,0xffc4a7236178378e72c22dfada4e9ed59c4339eb,BUY,2026-01-01 06:58:23+00:00,0x50a8b708290c3effad10f8ee36e104fa3c0c593b8ace...,Calgary Wranglers,433.000000,433.000000,0.370000,True,1.0,160.210000,433.000000,272.790000,1


## 8 · Export: apply PnL cut-off and write partitioned parquet

Filter grouped trades to ``final_wallets`` (wallets above the median per-shard PnL),
then write one parquet shard per first hex character of ``condition_id`` after ``0x``.

In [12]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 925


In [13]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  389,972
  training rows: 246,524
  test rows:     143,448
Output shards:  16
  0.parquet  (17,852 rows)
  1.parquet  (22,223 rows)
  2.parquet  (25,818 rows)
  3.parquet  (29,904 rows)
  4.parquet  (21,364 rows)
  5.parquet  (17,350 rows)
  6.parquet  (30,779 rows)
  7.parquet  (26,064 rows)
  8.parquet  (18,745 rows)
  9.parquet  (20,851 rows)
  a.parquet  (27,069 rows)
  b.parquet  (18,607 rows)
  c.parquet  (30,243 rows)
  d.parquet  (23,046 rows)
  e.parquet  (34,123 rows)
  f.parquet  (25,934 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [14]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.

In [15]:
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")

INFO  CTF Exchange rows not present in current filtered dataset
PASS  CTF Exchange not in top_wallets: top_wallets has 925 entries
